In [1]:
import pandas as pd
import numpy as np

from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind

In [2]:
df = pd.read_csv("../Dataset/results.csv")

# Cleaning
df = df.dropna()
df["date"] = pd.to_datetime(df["date"])

# Feature Engineering
df["Year"] = df["date"].dt.year

bins = [1872,1892,1912,1932,1952,1972,1992,2012,2032]

labels=[
"1872-1891",
"1892-1911",
"1912-1931",
"1932-1951",
"1952-1971",
"1972-1991",
"1992-2011",
"2012-2031"
]

df["Era"]=pd.cut(df["Year"],bins=bins,labels=labels,right=False)

df["Goal_Diff"]=df["home_score"]-df["away_score"]

df["Result"]=df.apply(
lambda x:"Home Win" if x.home_score>x.away_score
else "Away Win" if x.home_score<x.away_score
else "Draw",
axis=1
)

df["Home_Win"]=(df["home_score"]>df["away_score"]).astype(int)

df["Match_Type"]=np.where(
df["tournament"]=="Friendly",
"Friendly",
"Official"
)

df["Neutral"]=df["neutral"].astype(int)

### آیا نوع مسابقه روی برد میزبان اثر دارد؟

In [3]:
contingency_table = pd.crosstab(
    df["Match_Type"],
    df["Home_Win"]
)

contingency_table

Home_Win,0,1
Match_Type,,
Friendly,9688,8700
Official,15548,15551


In [4]:
chi2,p,dof,expected = chi2_contingency(contingency_table)

print("Chi-Square:",chi2)
print("P-value:",p)
print("Degrees of Freedom:",dof)

Chi-Square: 33.386220091841246
P-value: 7.555743179790685e-09
Degrees of Freedom: 1


### آیا زمین بی‌طرف اثر دارد؟

In [5]:
neutral_table = pd.crosstab(
    df["Neutral"],
    df["Home_Win"]
)

neutral_table

Home_Win,0,1
Neutral,,
0,17907,18451
1,7329,5800


In [6]:
chi2,p,dof,expected = chi2_contingency(neutral_table)

print("Chi-Square:",chi2)
print("P-value:",p)

Chi-Square: 166.4039837540896
P-value: 4.51461230488879e-38


### آیا Era روی مزیت میزبانی اثر دارد؟

In [7]:
era_table = pd.crosstab(
    df["Era"],
    df["Home_Win"]
)

era_table

Home_Win,0,1
Era,,
1872-1891,38,42
1892-1911,129,121
1912-1931,600,711
1932-1951,924,1014
1952-1971,2526,2612
1972-1991,4863,4528
1992-2011,8903,8559
2012-2031,7253,6664


In [8]:
chi2,p,dof,expected = chi2_contingency(era_table)

print("Chi-Square:",chi2)
print("P-value:",p)

Chi-Square: 39.53811860032438
P-value: 1.5427145758954692e-06


In [9]:
def cramers_v(table):
    chi2 = chi2_contingency(table)[0]
    n = table.values.sum()
    r, k = table.shape
    return np.sqrt(chi2 / (n * (min(r-1, k-1))))

In [10]:
cramers_v(contingency_table)

np.float64(0.025973953734340013)

In [11]:
cramers_v(neutral_table)

np.float64(0.05798775511474394)

In [12]:
cramers_v(era_table)

np.float64(0.028265875159160264)

### جدول خلاصه

In [13]:
summary = pd.DataFrame({
    "Hypothesis": [
        "Match Type vs Home Win",
        "Neutral vs Home Win",
        "Era vs Home Win"
    ],
    "Chi-Square": [
        33.386220091841246,
        166.4039837540896,
        39.53811860032438
    ],
    "P-value": [
        7.555743179790685e-09,
        4.51461230488879e-38,
        1.5427145758954692e-06
    ],
    "Cramer's V": [
        cramers_v(contingency_table),
        cramers_v(neutral_table),
        cramers_v(era_table)
    ]
})

summary

,Hypothesis,Chi-Square,P-value,Cramer's V
0,Match Type vs Home Win,33.386220,7.555743e-09,0.025974
1,Neutral vs Home Win,166.403984,4.514612e-38,0.057988
2,Era vs Home Win,39.538119,1.542715e-06,0.028266
